# svrecon benchmark

330 SVs (22 types x 3 size classes x 5) simulated with insilicoSV, then scored by svrecon
under three CIGAR-validation modes, on a positive and a negative callset.

| arm | callset | assembly | expected |
|---|---|---|---|
| positive | seed 0 | seed 0 | 330/330 hits |
| negative | seed 1 | seed 0 | 0/330 hits |

Hits in the negative arm are false positives of that mode.

Configs live in `workflows/insilicoSV/` and `workflows/svrecon/`; this notebook only runs them.

**Prerequisites:** run from the svrecon repo root, with `insilicosv` and `svrecon` pip-installed.
Run all cells top to bottom; the last cell deletes everything generated.

In [ ]:
%%bash
set -euo pipefail
for t in insilicosv svrecon; do command -v $t >/dev/null || { echo "$t not on PATH"; exit 1; }; done
test -f workflows/insilicoSV/release/insilicoSV.yaml || { echo 'run me from the svrecon repo root'; exit 1; }
echo ok

## 1. Download inputs

hg38 chr21 and its RepeatMasker intervals. The rmsk table is genome-wide (~180 MB),
piped down to chr21 rows. Skips anything already present.

In [ ]:
%%bash
set -euo pipefail
mkdir -p workflows/data

[ -f workflows/data/chr21.fa ] || \
  curl -fSL https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr21.fa.gz \
  | gunzip -c > workflows/data/chr21.fa

# rmsk.txt columns: 6 genoName, 7 genoStart, 8 genoEnd, 12 repClass
[ -f workflows/data/chr21.rmsk.bed ] || \
  curl -fSL https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/rmsk.txt.gz \
  | gunzip -c \
  | awk -F'\t' -v OFS='\t' '$6=="chr21" {print $6, $7, $8, $12}' \
  | sort -k1,1 -k2,2n > workflows/data/chr21.rmsk.bed

ls -lh workflows/data
wc -l < workflows/data/chr21.rmsk.bed

## 2. Simulate

Writes `sim.vcf` / `sim.hapA.fa` beside each config. ~1 min.

In [ ]:
%%bash
set -euo pipefail
log=$(mktemp)
for arm in release release_neg; do
  (cd workflows/insilicoSV/$arm && insilicosv -c insilicoSV.yaml) >"$log" 2>&1 \
    || { tail -25 "$log"; rm -f "$log"; exit 1; }
  echo "$arm: $(grep -vc '^#' workflows/insilicoSV/$arm/sim.vcf) VCF records"
done
rm -f "$log"

## 3. Score

Six runs: 2 arms x 3 modes. Writes `svrecon.log` beside each config. ~10 min.

In [ ]:
%%bash
set -euo pipefail
log=$(mktemp)
for arm in release release_neg; do
  for mode in similarity-only similarity-no-large-indels similarity-junction; do
    (cd workflows/svrecon/$arm/$mode && svrecon --config svrecon.yaml) >"$log" 2>&1 \
      || { tail -25 "$log"; rm -f "$log"; exit 1; }
    echo "scored $arm $mode"
  done
done
rm -f "$log"

### Results

In [ ]:
import json, re
from collections import Counter, defaultdict
from pathlib import Path

WORKFLOW = Path('workflows')
ARMS = ['release', 'release_neg']
MODES = ['similarity-only', 'similarity-no-large-indels', 'similarity-junction']
SIZES = ['small', 'medium', 'large']

def size_classes(vcf):
    """SVID -> size class, from its longest interval."""
    lens = defaultdict(list)
    for line in open(vcf):
        if line.startswith('#'):
            continue
        info = dict(kv.split('=', 1) for kv in line.split('\t')[7].split(';') if '=' in kv)
        lens[info['SVID']].append(int(info['SVLEN']))
    return {k: 'small' if max(v) < 500 else 'medium' if max(v) < 5000 else 'large'
            for k, v in lens.items()}

def outcomes(log):
    """SVID -> outcome, from the JSON blob ending each verbose log line."""
    out = {}
    for line in open(log):
        m = re.search(r'(\{"svid".*\})\s*$', line)
        if m:
            rec = json.loads(m.group(1))
            out[rec['svid']] = rec['outcome']
    return out

CLS = {a: size_classes(WORKFLOW / 'insilicoSV' / a / 'sim.vcf') for a in ARMS}

rows = []
for arm in ARMS:
    for mode in MODES:
        out = outcomes(WORKFLOW / 'svrecon' / arm / mode / 'svrecon.log')
        tot = Counter(CLS[arm][k] for k in out)
        hit = Counter(CLS[arm][k] for k, v in out.items() if v == 'hit')
        rows.append(['positive' if arm == 'release' else 'negative', mode,
                     *[f'{hit[s]}/{tot[s]}' for s in SIZES],
                     f'{sum(hit.values())}/{len(out)}',
                     f'{sum(hit.values()) / len(out):.2f}'])

hdr = ['arm', 'mode', *SIZES, 'total', 'precision']
w = [max(len(str(r[i])) for r in [hdr] + rows) for i in range(len(hdr))]
line = lambda r: '  '.join(str(c).ljust(w[i]) for i, c in enumerate(r))
print(line(hdr))
print('  '.join('-' * x for x in w))
for r in rows:
    print(line(r))
print('\nnegative arm: hits are false positives, lower is better')